# Geração de Imagens e Treinamento Ângulo-Ordem (Order-CWT)

**Projeto:** Diagnóstico de Falhas em Rolamentos via CWT + Deep Learning  
**Metodologia:** Computed Order Tracking (COT) + Continuous Wavelet Transform  
**Objetivo:** Gerar escalogramas no domínio de ordens (amostras uniformes por volta do eixo) e treinar as arquiteturas convolucionais (`BearingCNN` e `ResNet-18`) com invariância à rotação (RPM).

---

## 1. Importação de Módulos e Configurações

In [ ]:
import sys
from pathlib import Path

BASE_DIR = Path.cwd().parent.parent if Path.cwd().name == '4_order_tracking' else Path.cwd().parent
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

import torch
import numpy as np
import matplotlib.pyplot as plt

from src.config import (
    ORDER_CWRU_DIR,
    ORDER_PADERBORN_DIR,
    NUM_EPOCHS,
    LEARNING_RATE,
    RANDOM_SEED
)
from src.generate_order_dataset import generate_cwru_order_dataset, generate_paderborn_order_dataset
from src.transfer_learning import train_and_evaluate_transfer_model
from src.cnn_processor import train_and_evaluate_bearing_cnn
from src.visualization import plot_training_curves, plot_confusion_matrix

## 2. Geração Automática dos Escalogramas Ângulo-Ordem (Order-CWT)

Gera os datasets em `data/order_tracking/cwru/` e `data/order_tracking/paderborn/` sem Data Leakage.

In [ ]:
# 1. Gerar escalogramas de ordens para o CWRU
generate_cwru_order_dataset()

# 2. Gerar escalogramas de ordens para o Paderborn (10 arquivos por classe para teste rápido)
generate_paderborn_order_dataset(max_files_per_class=10)

## 3. Treinamento da ResNet-18 no Dataset CWRU Ângulo-Ordem

In [ ]:
res_cwru_order = train_and_evaluate_transfer_model(
    model_name="resnet18",
    num_epochs=NUM_EPOCHS,
    lr=LEARNING_RATE,
    freeze=False,
    data_dir=ORDER_CWRU_DIR,
    checkpoint_prefix="checkpoint_order_cwru"
)

plot_training_curves(
    res_cwru_order['history'],
    model_name="ResNet-18 (CWRU Order-CWT)",
    save_path="../../docs/images/training_curves_order_cwru_resnet.png"
)

plot_confusion_matrix(
    res_cwru_order,
    model_name="ResNet-18 (CWRU Order-CWT)",
    save_path="../../docs/images/confusion_matrix_order_cwru_resnet.png"
)

## 4. Treinamento da ResNet-18 no Dataset Paderborn Ângulo-Ordem

In [ ]:
res_pad_order = train_and_evaluate_transfer_model(
    model_name="resnet18",
    num_epochs=NUM_EPOCHS,
    lr=LEARNING_RATE,
    freeze=False,
    data_dir=ORDER_PADERBORN_DIR,
    checkpoint_prefix="checkpoint_order_paderborn"
)

plot_training_curves(
    res_pad_order['history'],
    model_name="ResNet-18 (Paderborn Order-CWT)",
    save_path="../../docs/images/training_curves_order_paderborn_resnet.png"
)

plot_confusion_matrix(
    res_pad_order,
    model_name="ResNet-18 (Paderborn Order-CWT)",
    save_path="../../docs/images/confusion_matrix_order_paderborn_resnet.png"
)